In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import duckdb

In [2]:
import json

In [3]:
df = pd.read_json("barkley_detection_common.jsonl", orient="records", lines=True)

In [4]:
df.shape

(308865, 3)

In [5]:
df.head(10)

,instruction,path,answer
0,{barkley-detection-common},audio/barkley-detection/ICLISTENHF1251_2013052...,None
1,{barkley-detection-common},audio/barkley-detection/ICLISTENHF1251_2013052...,None
2,{barkley-detection-common},audio/barkley-detection/ICLISTENHF1251_2013052...,None
3,{barkley-detection-common},audio/barkley-detection/ICLISTENHF1251_2013052...,None
4,{barkley-detection-common},audio/barkley-detection/ICLISTENHF1251_2013052...,None
5,{barkley-detection-common},audio/barkley-detection/ICLISTENHF1251_2013052...,None
6,{barkley-detection-common},audio/barkley-detection/ICLISTENHF1251_2013052...,None
7,{barkley-detection-common},audio/barkley-detection/ICLISTENHF1251_2013052...,None
8,{barkley-detection-common},audio/barkley-detection/ICLISTENHF1251_2013052...,None
9,{barkley-detection-common},audio/barkley-detection/ICLISTENHF1251_2013052...,None


In [6]:
df["answer"].unique()

array(['None', 'Killer whale', 'Humpback whale', 'Fin whale',
       'Fin whale, Humpback whale', 'Sperm whale',
       'Fin whale, Sperm whale', 'Blue whale', 'Fin whale, Killer whale',
       'Risso’s dolphin', 'Humpback whale, Sperm whale',
       'Risso’s dolphin, Sperm whale', 'Humpback whale, Risso’s dolphin',
       'Pacific white-sided dolphin', 'Humpback whale, Killer whale',
       'Humpback whale, Pacific white-sided dolphin',
       'Killer whale, Sperm whale', 'Blue whale, Sperm whale',
       'Blue whale, Humpback whale', 'Blue whale, Risso’s dolphin',
       'Blue whale, Fin whale', 'Blue whale, Fin whale, Humpback whale',
       'Fin whale, Pacific white-sided dolphin',
       'Blue whale, Pacific white-sided dolphin',
       'Fin whale, Humpback whale, Pacific white-sided dolphin'],
      dtype=object)

# Add taxonomy

## Use taxonomy which was already computed in the other csv

In [7]:
bc = pd.read_csv("barkley_canyon_annotations_non_null_gbif_v2.csv")

In [8]:
bc.columns

Index(['Soundfile', 'Top freq (Hz)', 'Bottom freq (Hz)', 'Species',
       'Call Type', 'Ecotype', 'gbifID', 'kingdom', 'phylum', 'class', 'order',
       'family', 'genus', 'species_common', 'species_scientific',
       'taxonomic_name', 'local_path', 'start_times(sec)', 'end_times(sec)'],
      dtype='object')

In [38]:
bc.sample(n=50)

,Soundfile,Top freq (Hz),Bottom freq (Hz),Species,Call Type,Ecotype,gbifID,kingdom,phylum,class,order,family,genus,species_common,species_scientific,taxonomic_name,local_path,start_times(sec),end_times(sec)
3786,ICLISTENHF1251_20140401T165458.941Z.wav,2577.61,770.80,Oo,BP,NaN,[2440483],Animalia,Chordata,Mammalia,Cetacea,Delphinidae,Orcinus,Killer whale,Orcinus orca,Chordata Mammalia Cetacea Delphinidae Orcinus ...,audio/ICLISTENHF1251_20140401T165458.941Z.wav,263.650190,265.285160
6704,ICLISTENHF1251_20140904T125409.210Z.wav,37.94,17.03,UN|Bm,NaN,NaN,[2440735],Animalia,Chordata,Mammalia,Cetacea,Balaenopteridae,Balaenoptera,Blue whale,Balaenoptera musculus,Chordata Mammalia Cetacea Balaenopteridae Bala...,audio/ICLISTENHF1251_20140904T125409.210Z.wav,104.752850,113.155890
3054,ICLISTENHF1251_20140204T135224.865Z.wav,487.20,245.17,Mn,S,NaN,[9809568],Animalia,Chordata,Mammalia,Cetacea,Balaenopteridae,Megaptera,Humpback whale,Megaptera novaeangliae,Chordata Mammalia Cetacea Balaenopteridae Mega...,audio/ICLISTENHF1251_20140204T135224.865Z.wav,16.958160,19.315590
4769,ICLISTENHF1251_20140702T003216.335Z.wav,773.10,62.90,UN|Mn,NaN,NaN,[9809568],Animalia,Chordata,Mammalia,Cetacea,Balaenopteridae,Megaptera,Humpback whale,Megaptera novaeangliae,Chordata Mammalia Cetacea Balaenopteridae Mega...,audio/ICLISTENHF1251_20140702T003216.335Z.wav,103.917000,104.733000
8306,ICLISTENHF1251_20141104T140031.845Z.wav,770.80,366.48,Mn,S,NaN,[9809568],Animalia,Chordata,Mammalia,Cetacea,Balaenopteridae,Megaptera,Humpback whale,Megaptera novaeangliae,Chordata Mammalia Cetacea Balaenopteridae Mega...,audio/ICLISTENHF1251_20141104T140031.845Z.wav,25.019010,26.463880
1720,ICLISTENHF1251_20140103T181200.449Z.wav,572.17,204.13,Mn,S,NaN,[9809568],Animalia,Chordata,Mammalia,Cetacea,Balaenopteridae,Megaptera,Humpback whale,Megaptera novaeangliae,Chordata Mammalia Cetacea Balaenopteridae Mega...,audio/ICLISTENHF1251_20140103T181200.449Z.wav,134.346950,135.934350
3280,ICLISTENHF1251_20140301T173327.586Z.wav,8958.02,733.74,Oo,NaN,NaN,[2440483],Animalia,Chordata,Mammalia,Cetacea,Delphinidae,Orcinus,Killer whale,Orcinus orca,Chordata Mammalia Cetacea Delphinidae Orcinus ...,audio/ICLISTENHF1251_20140301T173327.586Z.wav,111.071910,113.315250
4061,ICLISTENHF1251_20140501T000349.292Z.wav,8389.51,527.22,Pm,NaN,NaN,[8123917],Animalia,Chordata,Mammalia,Cetacea,Physeteridae,Physeter,Sperm whale,Physeter macrocephalus,Chordata Mammalia Cetacea Physeteridae Physete...,audio/ICLISTENHF1251_20140501T000349.292Z.wav,105.209130,106.045630
1877,ICLISTENHF1251_20140104T053200.573Z.wav,24638.34,6778.43,DE|Lb|Lo,BP,NaN,"[2440467, 5220069]","Animalia,Animalia","Chordata,Chordata","Mammalia,Mammalia","Cetacea,Cetacea","Delphinidae,Delphinidae","Lissodelphis,Lagenorhynchus","Northern right whale dolphin,Pacific white-sid...","Lissodelphis borealis,Lagenorhynchus obliquidens","Chordata,Chordata Mammalia,Mammalia Cetacea,Ce...",audio/ICLISTENHF1251_20140104T053200.573Z.wav,79.266263,79.742810
294,ICLISTENHF1251_20130623T075500.116Z.wav,5959.28,678.75,Oo,BP,BKW,[2440483],Animalia,Chordata,Mammalia,Cetacea,Delphinidae,Orcinus,Killer whale,Orcinus orca,Chordata Mammalia Cetacea Delphinidae Orcinus ...,audio/ICLISTENHF1251_20130623T075500.116Z.wav,208.609910,209.767940


In [13]:
import pdb
from tqdm.notebook import tqdm

In [14]:
taxonomies = []

for i, row in tqdm(df.iterrows(), total=len(df)):

    ans = row["answer"].split(",")

    tax = {"gbifID": None,
           "kingdom": None,
           "phylum": None,
           "class": None,
           "order": None,
           "family": None,
           "genus": None,
           "taxonomic_name": None
          }
    if len(ans) == 1 and ans[0] == "None":
        taxonomies.append(tax)
        continue
           
    for a in ans:
        if a == "None":
            continue
            
        matching_rows = bc.loc[bc["species_common"].apply(lambda x: a in x)]
        
        if matching_rows.empty:
            continue

        r = matching_rows.iloc[0].to_dict()

        for key in list(tax.keys()):
            if tax[key] is None:
                if key == "gbifID":
                    v = json.loads(r[key])
                    tax[key] = v if isinstance(v, list) else [v]
                else:
                    tax[key] = [r[key]]
            else:
                if key == "gbifID":
                    v = json.loads(r[key])
                    tax[key].extend(v)
                else:
                    tax[key].append(r[key])

    taxonomies.append(tax)

  0%|          | 0/308865 [00:00<?, ?it/s]

In [15]:
len(taxonomies)

308865

In [16]:
taxonomies[1000]

{'gbifID': [2440483],
 'kingdom': ['Animalia'],
 'phylum': ['Chordata'],
 'class': ['Mammalia'],
 'order': ['Cetacea'],
 'family': ['Delphinidae'],
 'genus': ['Orcinus'],
 'taxonomic_name': ['Chordata Mammalia Cetacea Delphinidae Orcinus orca']}

In [17]:
taxonomies = pd.DataFrame(taxonomies)
taxonomies.shape

(308865, 8)

In [18]:
taxonomies.head(3)

,gbifID,kingdom,phylum,class,order,family,genus,taxonomic_name
0,None,None,None,None,None,None,None,None
1,None,None,None,None,None,None,None,None
2,None,None,None,None,None,None,None,None


In [19]:
df = pd.concat([df.reset_index(drop=True), taxonomies], axis=1)
df.shape

(308865, 11)

## Add species_scientific

In [20]:
species_scientific = []
for i, row in df.iterrows():
    sci = []
    if row["family"] and len(row["family"]) > 0:
        if len(row["family"]) > 1:        
            pdb.set_trace()
            
        for r in row["taxonomic_name"]:
            s = " ".join(r.split(" ")[-2:])
            s = s.strip()
            sci.append(s)
            
    else:
        sci = None

    species_scientific.append(sci)

In [21]:
df["species_scientific"] = species_scientific

# Add file_name column

In [22]:
from pathlib import Path

In [23]:
df["file_name"] = df["path"].apply(lambda x: Path(x).name)

In [24]:
df["file_name"].head(3)

0    ICLISTENHF1251_20130520T003321.072Z_000.wav
1    ICLISTENHF1251_20130520T003321.072Z_001.wav
2    ICLISTENHF1251_20130520T003321.072Z_002.wav
Name: file_name, dtype: object

# Change 'path' to local_path and remove 'barkley-detection' from it

In [25]:
df.rename(columns={"path": "local_path"}, inplace=True)
df["local_path"] = df["local_path"].apply(lambda x: Path("audio") / Path(x).name)

In [26]:
df["local_path"].head(5)

0    audio/ICLISTENHF1251_20130520T003321.072Z_000.wav
1    audio/ICLISTENHF1251_20130520T003321.072Z_001.wav
2    audio/ICLISTENHF1251_20130520T003321.072Z_002.wav
3    audio/ICLISTENHF1251_20130520T003321.072Z_003.wav
4    audio/ICLISTENHF1251_20130520T003321.072Z_004.wav
Name: local_path, dtype: object

# Rename 'answer' to species_common

In [27]:
df.rename(columns={"answer": "species_common"}, inplace=True)

In [28]:
df["species_common"].dtype

dtype('O')

### Convert it to list[str] instead of comma separated list

In [37]:
df.sample(n=50)

,instruction,local_path,species_common,gbifID,kingdom,phylum,class,order,family,genus,taxonomic_name,species_scientific,file_name
13116,{barkley-detection-common},audio/ICLISTENHF1251_20140102T040200.028Z_018.wav,None,None,None,None,None,None,None,None,None,None,ICLISTENHF1251_20140102T040200.028Z_018.wav
25581,{barkley-detection-common},audio/ICLISTENHF1251_20140103T163200.428Z_034.wav,None,None,None,None,None,None,None,None,None,None,ICLISTENHF1251_20140103T163200.428Z_034.wav
105450,{barkley-detection-common},audio/ICLISTENHF1251_20140403T025459.324Z_017.wav,None,None,None,None,None,None,None,None,None,None,ICLISTENHF1251_20140403T025459.324Z_017.wav
282625,{barkley-detection-common},audio/ICLISTENHF1251_20141201T215144.815Z_015.wav,None,None,None,None,None,None,None,None,None,None,ICLISTENHF1251_20141201T215144.815Z_015.wav
43217,{barkley-detection-common},audio/ICLISTENHF1251_20140201T202456.169Z_029.wav,None,None,None,None,None,None,None,None,None,None,ICLISTENHF1251_20140201T202456.169Z_029.wav
79514,{barkley-detection-common},audio/ICLISTENHF1251_20140302T121327.796Z_041.wav,None,None,None,None,None,None,None,None,None,None,ICLISTENHF1251_20140302T121327.796Z_041.wav
86883,{barkley-detection-common},audio/ICLISTENHF1251_20140304T045328.231Z_035.wav,None,None,None,None,None,None,None,None,None,None,ICLISTENHF1251_20140304T045328.231Z_035.wav
90004,{barkley-detection-common},audio/ICLISTENHF1251_20140304T144328.348Z_029.wav,None,None,None,None,None,None,None,None,None,None,ICLISTENHF1251_20140304T144328.348Z_029.wav
304843,{barkley-detection-common},audio/ICLISTENHF1251_20141204T123118.993Z_049.wav,None,None,None,None,None,None,None,None,None,None,ICLISTENHF1251_20141204T123118.993Z_049.wav
112290,{barkley-detection-common},audio/ICLISTENHF1251_20140404T044459.579Z_013.wav,None,None,None,None,None,None,None,None,None,None,ICLISTENHF1251_20140404T044459.579Z_013.wav


# Save

In [30]:
df.columns

Index(['instruction', 'local_path', 'species_common', 'gbifID', 'kingdom',
       'phylum', 'class', 'order', 'family', 'genus', 'taxonomic_name',
       'species_scientific', 'file_name'],
      dtype='object')

In [31]:
df.to_csv("gs://esp-ml-datasets/barkley_canyon_detection/v0.1.0/raw/barkley_canyon_detection_gbif.csv", index=False, storage_options={"project": "okapi-274503"})

In [33]:
df.to_json("gs://esp-ml-datasets/barkley_canyon_detection/v0.1.0/raw/barkley_canyon_detection_gbif.jsonl", index=False, orient="records", lines=True, storage_options={"project": "okapi-274503"})

OverflowError: Maximum recursion level reached